**开始前：怎样找到章节和自测**

在 JupyterLab 上方菜单选择 **View → Table of Contents**（中文界面通常为“视图 → 目录”，名称可能随版本略有不同），打开左侧目录；点击目录中的章节标题、互动实验或“自测 7：模型评估”，就能跳到对应位置。想回到开头时，点击目录中的“从挑西瓜到决策树、随机森林与梯度提升”。

第一次学习建议从头按顺序运行：先认识单棵树，再学习随机森林、回归树、梯度提升和 XGBoost，最后比较模型并完成自测 7。**点击目录只会移动位置，不会运行代码。**跳到某一节之前，请先顺序运行前面的准备和数据单元；重启内核后也需要重新运行。标为“选学”或“自测”的作答可以跳过。


# 从挑西瓜到决策树、随机森林与梯度提升

挑西瓜时，我们会看**纹理、瓜蒂、色泽**等特征，想判断它是不是好瓜。周志华《机器学习》用“西瓜数据集”帮助读者理解机器学习；本章借用这个容易想象的问题：除下一格注明的教材摘录外，其余数据、简化规则和结果均为教学**自行编造**，不是挑瓜建议。[作者的书籍主页](https://cs.nju.edu.cn/zhouzh/zhouzh.files/publication/MLbook2016.htm)

![从纹理和瓜蒂开始逐步判断好瓜的示意树](../assets/watermelon_tree_path.png)

上图是一棵**示意树**：从上往下，每个框问一个问题，按回答走一条路，到底部得到预测。如果纹理清晰、瓜蒂蜷缩，就沿“是→是”走到“预测好瓜”；纹理不清晰则直接到另一个叶子。它只是示意，并非真实规则，也不是下面数据训练出来的树。

这一章先学**一棵树怎样提出问题**，再比较许多树如何分别投票，或一棵接一棵修正。为理解回归与提升，后半章仍研究西瓜，不过将目标换成模拟甜度。按顺序运行，看到「自测」可选择作答或跳过。

### 先看教材中的几条记录

周志华《机器学习》的西瓜数据集 2.0 用已知的西瓜特征与“好瓜”标签说明分类。下表只摘录其中四条及三个特征，保留原编号，便于认识这个问题；[清华大学出版社《Python机器学习项目化教程》公开样章表 3-1](https://www.tup.tsinghua.edu.cn/upload/books/yz/099994-01.pdf)引用并列出了该数据集，供核对。

| 编号 | 色泽 | 根蒂 | 纹理 | 好瓜 |
| --- | --- | --- | --- | --- |
| 1 | 青绿 | 蜷缩 | 清晰 | 是 |
| 7 | 乌黑 | 稍蜷 | 稍糊 | 是 |
| 10 | 青绿 | 硬挺 | 清晰 | 否 |
| 16 | 浅白 | 蜷缩 | 模糊 | 否 |

从少量记录可看到：好瓜与坏瓜都有“清晰”纹理，仅凭一个特征不一定能判断。下面的图和代码另用**虚构教学数据**，不要把它们当作教材原表或真实挑瓜依据。


## 1. 如何选第一个问题？

假设手头有 6 个已经知道好坏的瓜。我们考虑先按**色泽**还是**纹理**划分；目标是让同一组里的结果尽量一致。

![六个模拟西瓜，比较色泽和纹理的两种候选分裂](../assets/watermelon_split_mechanics.png)

这 6 个瓜是独立的模拟例子，不是教材的 6 条记录。按色泽分，两组各有好瓜和坏瓜；按纹理分后，一组只有坏瓜，另一组仍混有少量坏瓜。比较时不能只盯着其中一组：每组按瓜的数量计算权重，合在一起看。图中按纹理划分后的加权不纯度为 0.25，比按色泽的 4/9 更小。

**训练**时，我们用已知好坏比较候选问题；**预测**新瓜时，它的好坏尚未知晓，只用特征走已经学好的分支。

### 把“混杂”变成一个数：基尼不纯度

一个组里全是好瓜或全是坏瓜，就不混杂，基尼为 0；好坏各半时基尼为 0.5。对两类瓜，若好瓜占 $p$、坏瓜占 $1-p$，则

$$G=1-p^2-(1-p)^2,\qquad G_{split}=\frac{n_L}{n}G_L+\frac{n_R}{n}G_R$$

后一个式子是把两组的基尼按数量加权。对同一批瓜比较候选问题，结果越小，说明两组合起来越不混杂。下面的函数和图使用同一组 6 个模拟瓜。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

# 与上图一致：好瓜=1，坏瓜=0；非教材原始数据。
toy_good = np.array([1, 1, 0, 1, 0, 0])
toy_color = np.array([0, 0, 0, 1, 1, 1])   # 0=青，1=黑
toy_texture = np.array([1, 1, 1, 1, 0, 0]) # 1=清晰，0=模糊

**自测准备**：运行下一格启用后面的单选题；无需修改代码。每题可跳过作答，不影响模型和后续课程。

In [ ]:
# 【自测准备】运行即可。每题的作答和反馈相互独立。
from html import escape as _escape
_quiz_bank = {
    'split': ('同一批瓜按色泽划分的加权基尼为 4/9，按纹理划分为 0.25。若只按这个标准选，选哪一个？', [
        ('色泽：4/9 较大', '基尼衡量混杂；比较同一父节点时，较小的结果更纯。'),
        ('纹理：0.25 较小', '两组按样本数加权后，0.25 小于 4/9，因此纹理更纯。'),
        ('只看左组，不能看整体', '必须考虑左右所有瓜，按组大小加权。')], 1),
    'overfit': ('树越来越深时训练准确率提高，验证准确率却下降，说明什么？', [
        ('可能记住了训练数据中的偶然细节', '这是过拟合的常见信号，可以比较限制深度或剪枝。'),
        ('证明所有未来瓜都会判断错误', '一次验证结果有波动，不能证明所有未来样本都一样。'),
        ('训练表现好就不需要验证', '训练集已用于学习，不能代替未参与拟合的数据。')], 0),
    'forest': ('三棵树对同一个瓜给出的好瓜概率是 0.75、0.5、0.25。按概率平均，森林给出的概率是多少？', [
        ('0.5', '三数平均为 (0.75 + 0.5 + 0.25) / 3 = 0.5。'),
        ('1.5', '1.5 是和；需要除以三棵树。'),
        ('0.75', '只取最高的树并不是平均；三棵树都要参与。')], 0),
    'regression': ('一片回归叶子里的模拟甜度分别是 6 和 8，使用平均值预测时输出多少？', [
        ('6', '取两瓜的平均，不是只取第一个。'), ('7', '(6 + 8) / 2 = 7。'), ('14', '14 是总和，还要除以数量 2。')], 1),
    'boost': ('当前预测甜度 7，新树输出 4，学习率 0.3。更新后的预测是多少？', [
        ('11', '先将新树输出乘学习率，再加到原预测，不能直接全加上。'),
        ('8.2', '7 + 0.3 × 4 = 8.2；新树输出的是修正量。'),
        ('1.2', '1.2 是修正量，还要加上原预测 7。')], 1),
    'xgb': ('固定非零 G、正 H，只考虑 L2 正则；w = −G / (H + λ)。增大 λ 会怎样？', [
        ('叶权绝对值变大', '分母增大时，固定非零分子所得比值的绝对值会变小。'),
        ('叶权绝对值变小', 'λ 增大使分母增大，因而修正更保守；G 为 0 时叶权始终为 0。'),
        ('叶权必然为负', '符号由 −G 决定，与 λ 的增大无关。')], 1),
    'evaluation': ('比较并选择模型后，独立测试集怎样用？', [
        ('每改一次参数都用它挑最好结果', '反复根据测试结果挑选会让它参与选择，失去独立评估作用。'),
        ('验证集选择方案，最后用测试集评估', '训练集学参数，验证集选方案；测试集在选择后评估。'),
        ('只用训练集就够', '训练集参与拟合，不足以评估未见样本。')], 1),
}
def lesson_quiz(key):
    question, options, correct = _quiz_bank[key]
    title = widgets.HTML(value='<b>' + _escape(question) + '</b>')
    choices = widgets.RadioButtons(options=[(x[0], i) for i, x in enumerate(options)], value=None, layout=widgets.Layout(width='100%'))
    button = widgets.Button(description='提交答案', button_style='primary')
    feedback = widgets.HTML(value='')
    def clear(change): feedback.value = ''
    def check(_):
        if choices.value is None:
            feedback.value = '请先选择一个答案，再提交。'; return
        i = choices.value
        feedback.value = ('<b>回答正确。</b> ' if i == correct else '<b>再想一想。</b> ') + _escape(options[i][1])
    choices.observe(clear, names='value'); button.on_click(check)
    return widgets.VBox([title, choices, button, feedback])

In [ ]:
# 【练习】通过已有标签计算不纯度和一个问题的分裂结果。
def gini(labels):
    labels = np.asarray(labels)
    if not len(labels): raise ValueError('不能计算空组的不纯度')
    _, counts = np.unique(labels, return_counts=True)
    return 1 - np.sum((counts / len(labels)) ** 2)

def split_gini(left, labels):
    left, labels = np.asarray(left, dtype=bool), np.asarray(labels)
    if not left.any() or left.all(): raise ValueError('两边都需要有瓜')
    return left.mean() * gini(labels[left]) + (~left).mean() * gini(labels[~left])

assert np.isclose(split_gini(toy_color == 0, toy_good), 4/9)
assert np.isclose(split_gini(toy_texture == 1, toy_good), .25)
print('色泽:', round(split_gini(toy_color == 0, toy_good), 3),
      '；纹理:', round(split_gini(toy_texture == 1, toy_good), 3))

### 动手比较两个问题

下拉选择问题，左边代表回答“是”的瓜，右边代表“否”。瓜的颜色代表已知好坏，不是西瓜外皮颜色；先猜，再看加权基尼。

In [ ]:
# 【演示】直接运行并切换问题；图中圆点只是六个模拟瓜。
def show_split(question):
    left = (toy_color == 0) if question == '色泽是青？' else (toy_texture == 1)
    fig, ax = plt.subplots(figsize=(7, 2.8))
    for i in range(6):
        ax.scatter(0 if left[i] else 1, i, s=170,
                   color='#2a9d8f' if toy_good[i] else '#e76f51', edgecolor='white')
        ax.annotate(chr(65+i), (0 if left[i] else 1, i), ha='center', va='center', color='white')
    ax.set(xticks=[0, 1], xticklabels=['Yes', 'No'], yticks=[], xlim=(-.7, 1.7),
           title=f'Weighted Gini = {split_gini(left, toy_good):.3f}')
    ax.text(.02, -.12, 'Teal = good; coral = bad', transform=ax.transAxes)
    plt.show()
display(widgets.interactive(show_split, question=widgets.Dropdown(options=['色泽是青？', '纹理清晰？'], description='问题')))

### 自测 1：选问题

选出答案并提交；改变答案后可以重试。

In [ ]:
display(lesson_quiz('split'))

## 2. 一个问题如何长成一棵树？

在当前一组瓜中，决策树大致重复四件事：

1. 列出候选特征与问题，比如“纹理清晰吗？”；
2. 看每个问题把瓜分成哪些组，计算划分后的混杂程度；
3. 选当前最合适的问题，并对各组继续重复；
4. 不能再有效划分，或达到预设限制时停止；每片叶子保存该组的预测结果。

这叫**递归**：新组再次做同样的事情。分类叶子可用其中最多的类别作预测；如果 10 个训练瓜里有 7 个好瓜，该叶子的好瓜比例就是 0.7，但这不是对下一个瓜的保证。

下面不使用教材真实记录，而用 600 个**模拟瓜**练习完整流程。我们人为生成三个输入：纹理、瓜蒂、密度，再模拟好坏标签和甜度。为了让示例有规律，标签与特征有关，并加入随机波动；现实中不应把这些规律当作挑瓜依据。

In [ ]:
rng = np.random.default_rng(24)
n = 600
texture = rng.integers(0, 3, n)  # 0=模糊，1=稍糊，2=清晰（教学编码）
stem = rng.integers(0, 3, n)     # 0=硬挺，1=稍蜷，2=蜷缩（教学编码）
density = rng.uniform(.2, .8, n)
X = np.column_stack([texture, stem, density])
# 模拟标签和甜度；甜度不是输入，也不是用标签直接算出的。
score = -.9 + .8*texture + .3*stem + .7*(density-.5) + rng.normal(0, .9, n)
y_good = (score > .45).astype(int)
y_sweet = 4 + 5*density + .6*texture + .3*stem + rng.normal(0, .8, n)
# 测试集暂时封存；训练集学习，验证集探索与选择。
X_pool, X_test, good_pool, good_test, sweet_pool, sweet_test = train_test_split(
    X, y_good, y_sweet, test_size=.2, random_state=42, stratify=y_good)
X_train, X_valid, good_train, good_valid, sweet_train, sweet_valid = train_test_split(
    X_pool, good_pool, sweet_pool, test_size=.25, random_state=11, stratify=good_pool)
small_tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, good_train)
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(small_tree, feature_names=['Texture', 'Stem', 'Density'], class_names=['Bad', 'Good'], filled=True, rounded=True, ax=ax)
plt.show()
print(f'训练准确率 {small_tree.score(X_train, good_train):.3f}；验证准确率 {small_tree.score(X_valid, good_valid):.3f}')

图里的 `Texture`、`Stem`、`Density` 分别是纹理编码、瓜蒂编码和密度。每个条件成立走左边，不成立走右边。`samples` 是训练瓜数量，`gini` 是混杂程度；`value` 是类别分布（不同版本可能显示数量或比例）。这里给纹理和瓜蒂排序编码，只为了讲清演示，不表示所有类别特征都有自然顺序。

### 互动实验：亲手建立一棵决策树

先选择一个待处理节点，再试试“纹理、色泽或瓜蒂”三个问题。**分组图是预览**：每颗瓜都带着编号和好坏标记，看看问题如何把它们分成“是”和“否”两组，加权基尼又怎样变化。

决定好后，点击“用这个问题分裂”，树图才会长出分支。继续选择待处理的节点，直到每一组都定为叶子；再改变新瓜的特征，看高亮路径怎样穿过你建的树。实验沿用前面的六个模拟瓜，并额外添加模拟瓜蒂。运行下格即可开始；实验可跳过，不影响后续课程。

In [ ]:
# 【互动】运行后直接在下方操作；与后面的模型训练互不影响。
try:
    from MachineLearning.watermelon_tree_lab import watermelon_tree_lab
except ModuleNotFoundError as exc:
    if exc.name != 'MachineLearning':
        raise
    from watermelon_tree_lab import watermelon_tree_lab
display(watermelon_tree_lab())

## 3. 树越深越好吗？

一棵树如果一直问下去，可能把训练时每个瓜都记住，连偶然差异也当成规律。遇到新瓜时，这些细节未必适用，这叫**过拟合**。

拖动深度上限，比较训练和验证准确率；限制深度叫预剪枝。另一种方法是先让树长大，再用 `ccp_alpha` 等规则做后剪枝。验证结果也会随数据划分波动，不能只凭一次结果断言以后都如此。

In [ ]:
def show_depth(depth):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, good_train)
    values = [tree.score(X_train, good_train), tree.score(X_valid, good_valid)]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(['Train', 'Validation'], values, color=['#2a9d8f', '#e76f51'])
    ax.set(ylim=(0, 1.05), ylabel='Accuracy', title=f'Max depth = {depth}; leaves = {tree.get_n_leaves()}')
    for i, v in enumerate(values): ax.text(i, v+.02, f'{v:.3f}', ha='center')
    plt.show()
display(widgets.interactive(show_depth, depth=widgets.IntSlider(value=2, min=1, max=15, description='深度', continuous_update=False)))

In [ ]:
# 【练习】修改 ccp_alpha，比较叶子数和验证结果。
full_tree = DecisionTreeClassifier(random_state=42).fit(X_train, good_train)
pruned_tree = DecisionTreeClassifier(ccp_alpha=.01, random_state=42).fit(X_train, good_train)
for name, tree in [('Full', full_tree), ('Pruned', pruned_tree)]:
    print(f'{name:6s} 叶子 {tree.get_n_leaves():3d}  训练 {tree.score(X_train, good_train):.3f}  验证 {tree.score(X_valid, good_valid):.3f}')

### 自测 2：观察过拟合

In [ ]:
display(lesson_quiz('overfit'))

## 4. 随机森林：让多棵树一起判断

单棵树可能很依赖看到的训练瓜。一个想法是：从同一批瓜中抽出几份**略有不同的训练样本**，分别训练树，再合并它们的判断。

![四个模拟瓜有放回抽样，三棵树输出概率并求平均](../assets/watermelon_forest_bootstrap.png)

图里每次抽一个瓜后把它放回，因此下一次可以抽到同一个：例如 4 次抽样得到 A、A、B、C，D 就没有被抽到。各棵树分别使用一份这样的样本；**随机森林**还在每个节点随机挑选部分候选特征，避免所有树总用同一个特征。最后回归取平均；分类可以投票，具体实现也可能先平均各树概率。

图中三棵树分别给同一个新瓜 0.75、0.5、0.25 的好瓜概率，平均是 0.5。平均不同判断通常有助于降低波动，但不能消除共同的偏差，也不保证一定比单棵树准确。

In [ ]:
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=-1)
forest.fit(X_train, good_train)
example = np.array([[2., 1., .55]])  # 新瓜：纹理清晰、瓜蒂稍蜷、密度 0.55；特征只是模拟
votes = np.array([tree.predict(example)[0] for tree in forest.estimators_])
print(f'单棵深树验证准确率：{full_tree.score(X_valid, good_valid):.3f}')
print(f'森林验证准确率：{forest.score(X_valid, good_valid):.3f}')
print(f'100 棵树中预测好瓜的票数：{int(votes.sum())}；预测坏瓜：{100-int(votes.sum())}')
print(f'森林平均好瓜概率：{forest.predict_proba(example)[0, 1]:.3f}')

这里 scikit-learn 的分类森林先平均各树的**叶子类别概率**，再选择平均概率最高的类别；这个值不一定等于各树硬投票中好瓜的比例。图表使用模拟数据，不能推论真实挑瓜效果。

### 互动实验：自己组一片小森林

对每棵小树，从六个模拟瓜中选择四次有放回抽样，允许同一个瓜被选中多次；再选择一个问题。观察图中抽到的瓜、新瓜到达的叶子以及各棵树的预测，建立三棵树后看看平均概率如何变化。

这里为了看清过程，每棵树只问一个由你选择的问题；实际随机森林通常在节点随机抽取候选特征，再从中选划分。运行下格开始，实验可跳过。

In [ ]:
# 【互动】运行后直接在下方操作；与后面的模型训练互不影响。
try:
    from MachineLearning.watermelon_ensemble_labs import forest_lab
except ModuleNotFoundError as exc:
    if exc.name != 'MachineLearning':
        raise
    from watermelon_ensemble_labs import forest_lab
display(forest_lab())

### 自测 3：合并多棵树的判断

In [ ]:
display(lesson_quiz('forest'))

### 两种组合方式

![随机森林并行训练并聚合；梯度提升顺序增加修正树](../assets/watermelon_forest_vs_boosting.png)

森林中的树可以各自训练；另一条路是让后面的树看到前面模型的结果，**一轮一轮修正**。理解后一条路之前，先把“好瓜或坏瓜”换成数值预测：这瓜的模拟甜度是多少？

## 5. 回归树：从判断好坏到预测甜度

分类树输出类别；回归树输出数值。例如一个叶子里两瓜的甜度是 6 和 8，常见的叶子预测是平均值 7。如何选划分？可以比较分开以后，各组中的瓜与该组平均值相差多少。

![四个模拟瓜比较两种密度门槛和划分后的均方误差](../assets/watermelon_regression_split.png)

图中四个瓜模拟甜度为 6、8、10、12。按密度 0.5 分成 `[6, 8]` 和 `[10, 12]`，两组平均为 7 和 11，四个误差平方都是 1，平均为 1。按密度 0.3 分成 `[6]` 和 `[8, 10, 12]`，平均平方误差为 2。按这个标准，0.5 更好。

这只是一个候选比较。持续重复可以得到多片叶子，每片叶子对应一个预测值，因而预测图像看起来像台阶。

In [ ]:
# 为观察台阶，只用密度预测模拟甜度；其他两项暂时不用。
reg_tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train[:, [2]], sweet_train)
grid = np.linspace(.2, .8, 200).reshape(-1, 1)
plt.figure(figsize=(8, 4))
plt.scatter(X_train[:, 2], sweet_train, s=12, alpha=.25, label='Simulated training melons')
plt.plot(grid[:, 0], reg_tree.predict(grid), color='#e76f51', lw=3, label='Tree prediction')
plt.xlabel('Density'); plt.ylabel('Simulated sweetness'); plt.legend(); plt.grid(alpha=.2); plt.show()
print(f'只用密度时的验证均方误差：{mean_squared_error(sweet_valid, reg_tree.predict(X_valid[:, [2]])):.3f}')

只用一个特征的结果，不宜与后面使用三个特征的模型直接排名。均方误差越小表示该批瓜整体预测更接近模拟甜度，单位是甜度单位的平方。

### 自测 4：回归叶子的输出

In [ ]:
display(lesson_quiz('regression'))

## 6. 梯度提升树：一轮一轮修正甜度

随机森林让树分别预测，再合并。现在换个办法：先给出一个粗略甜度预测，再让下一棵树学习**当前预测还差多少**。

对于平方误差回归，实际甜度减去当前预测，叫**残差**。残差为正说明预测偏低，为负说明预测偏高。新树不从头预测总甜度，而是尝试预测修正量；把修正量乘一个学习率，再加到旧预测上。

![三瓜甜度两轮提升，展示残差变化与树的修正](../assets/watermelon_gbdt_rounds.png)

跟着图算：三瓜甜度是 6、8、10，先全部预测平均值 8，残差是 −2、0、2。第一棵树拟合残差，输出 −2、1、1；学习率是 0.5，更新后预测为 7、8.5、8.5，**新的残差**为 −1、−0.5、1.5。第二棵树再拟合这一组新的残差；图中第二轮的输出为 −0.75、−0.75、1.5。三瓜及树桩都是手算示例。

$$\text{新预测}=\text{旧预测}+\text{学习率}\times\text{新树的输出}$$

为什么叫“梯度”提升？可以把梯度先理解为“当前预测往哪一边移动，会使损失增大”；**负梯度**指向降低损失的方向。平方误差下，它与“实际值减预测值”的残差相同或只差固定倍数，所以可以用残差解释；换成分类等其他损失时，修正方向要依该损失计算，不能一律当作甜度残差。

### 为什么每轮的划分问题可能不同？

前面的三瓜密度按大小排列为 0.2、0.5、0.8。若只用密度问一次“是否不超过某个值”，可以先试相邻密度的中点：**0.35** 和 **0.65**。每次用一个问题分组，再计算各组内“残差离该组平均残差多远”。下面是这两个候选在**同一批瓜**上拟合残差的平方误差之和，越小表示该轮拟合残差越好：

| 比较时刻 | 当前残差 | 按 0.35 分组的误差和 | 按 0.65 分组的误差和 |
| --- | --- | --- | --- |
| 第一轮之前 | −2、0、2 | 2 | 2 |
| 第一轮采用 0.35、学习率 0.5 之后 | −1、−0.5、1.5 | 2 | 0.125 |

第一轮两个候选打平；图中选择 0.35。更新预测后，残差变了，再比较时 0.65 更小，因此图中的第二棵树选 0.65。**这不是说第二轮必须换问题**：如果仍选 0.35，也能计算更新；互动实验允许你故意这样做，观察差异。标准训练通常会按当前目标比较候选，而不是随机决定门槛；不同数据、参数或平分处理可能得到不同结果。

In [ ]:
# 【演示】训练一次，滑块查看加到不同树数时的结果。
boost = GradientBoostingRegressor(n_estimators=100, learning_rate=.08, max_depth=2, random_state=42)
boost.fit(X_train, sweet_train)
train_stages = list(boost.staged_predict(X_train)); valid_stages = list(boost.staged_predict(X_valid))
train_mse = [mean_squared_error(sweet_train, z) for z in train_stages]
valid_mse = [mean_squared_error(sweet_valid, z) for z in valid_stages]
def show_boost(stage):
    fig, (ax, line) = plt.subplots(1, 2, figsize=(10, 3.5))
    pred = valid_stages[stage-1]
    ax.scatter(sweet_valid, pred, alpha=.45, s=18, color='#2a9d8f')
    lo, hi = min(sweet_valid.min(), pred.min()), max(sweet_valid.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], '--', color='#e76f51')
    ax.set(xlabel='Actual simulated sweetness', ylabel='Prediction', title=f'Validation: tree {stage}')
    line.plot(range(1, 101), train_mse, label='Train'); line.plot(range(1, 101), valid_mse, label='Validation')
    line.axvline(stage, color='#e76f51', linestyle='--'); line.set(xlabel='Trees', ylabel='MSE'); line.legend()
    for a in (ax,line): a.grid(alpha=.2)
    plt.tight_layout(); plt.show()
display(widgets.interactive(show_boost, stage=widgets.IntSlider(value=1, min=1, max=100, description='树数', continuous_update=False)))

观察训练和验证误差：多加一棵树不保证每个瓜、每一轮都更准，验证误差也可能先下降后上升。用验证集选择树数，测试集留到最后。

### 互动实验：让小树一轮轮修正

三颗模拟瓜已有真实甜度。先观察图中真实值、当前预测和残差，再比较两个密度问题，选择其中一个和学习率。每添加一棵小树，都看看图形、预测和残差怎样更新；你也可以重置并换一种选择。

实验只演示平方误差回归：每片叶子的修正量取该组当前残差的平均。运行下格开始，实验可跳过。

In [ ]:
# 【互动】运行后直接在下方操作；与后面的模型训练互不影响。
try:
    from MachineLearning.watermelon_ensemble_labs import boosting_lab
except ModuleNotFoundError as exc:
    if exc.name != 'MachineLearning':
        raise
    from watermelon_ensemble_labs import boosting_lab
display(boosting_lab())

### 自测 5：算一次甜度更新

In [ ]:
display(lesson_quiz('boost'))

### 选学：自己写一个树桩和两轮以上的修正

**树桩**是一棵只问一次问题的回归树。第一格找出最小平方误差的门槛；第二格每轮都把当前残差交给新树桩。两格可跳过，不影响后续课程。

In [ ]:
def fit_stump(x, target):
    x, target = np.asarray(x, dtype=float), np.asarray(target, dtype=float)
    unique = np.unique(x)
    if len(unique) < 2: raise ValueError('至少需要两个不同输入')
    best = None
    for threshold in (unique[:-1] + unique[1:]) / 2:
        left = x <= threshold
        lm, rm = target[left].mean(), target[~left].mean()
        loss = ((target[left]-lm)**2).sum() + ((target[~left]-rm)**2).sum()
        if best is None or loss < best[0]: best = (loss, threshold, lm, rm)
    return best[1:]
def predict_stump(x, stump):
    threshold, lm, rm = stump
    return np.where(np.asarray(x) <= threshold, lm, rm)
x_toy = np.array([.2, .5, .8]); sweet_toy = np.array([6., 8., 10.])
first = fit_stump(x_toy, sweet_toy - sweet_toy.mean())
np.testing.assert_allclose(first, (.35, -2., 1.))
print('第一棵树桩：门槛、左右修正量', first)

In [ ]:
def fit_boost_numpy(x, y, rounds=8, learning_rate=.5):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    base = float(y.mean()); prediction = np.full(len(y), base)
    stumps, losses = [], [np.mean((y-prediction)**2)]
    for _ in range(rounds):
        stump = fit_stump(x, y-prediction); stumps.append(stump)
        prediction += learning_rate * predict_stump(x, stump)
        losses.append(np.mean((y-prediction)**2))
    return base, stumps, np.array(losses)
rate = .5  # 试着改变；过大的学习率可能让误差上升
base, stumps, losses = fit_boost_numpy(x_toy, sweet_toy, rounds=8, learning_rate=rate)
np.testing.assert_allclose(stumps[0], first)
_, _, safe_losses = fit_boost_numpy(x_toy, sweet_toy, rounds=8, learning_rate=.5)
assert np.all(np.diff(safe_losses) <= 1e-10)
print('初始预测：', base, '；训练均方误差：', np.round(losses, 3))
print('第二棵树桩：', stumps[1])

## 7. XGBoost：如何决定修正量，以及值不值得分裂？

前面的小树拟合残差，再把修正加入原预测。XGBoost 的默认树模型仍然**一轮一轮加树**；这一节关心两个更具体的问题：一组瓜共用的修正量该是多少？分成两组带来的改善，是否值得多长一片叶子？

先不用记公式，把损失想成“预测错得有多严重”。下面以**平方损失回归**为例，数据均为虚构教学数字。为方便运算，本节定义单瓜损失为 $L=\frac12(\text{预测}-\text{真实})^2$；它与前面均方误差只差汇总方式与常数，不改变“越接近越好”的方向。

### 第一步：看一颗瓜，理解方向和弯曲程度

一颗瓜的真实甜度是 6，当前预测是 9。损失为 $\frac12(9-6)^2=4.5$；把预测改成 8 后，损失是 $\frac12(8-6)^2=2$，所以这里往下调是有利的。

![平方损失曲线：真实甜度为6，预测从9调到8时损失由4.5降到2；在预测9处的梯度为3](../assets/watermelon_xgb_gradient.png)

- **梯度 $g$** 表示当前这点的损失斜率：在预测 9 附近，$g=3$，预测略微增加会让损失增加；反方向略微减小有助于降低损失。它针对的是**损失**，不只是原始误差。
- **二阶信息 $h$** 表示斜率随预测改变得多快。在这个平方损失例子中，预测从 8 增到 9，$g$ 从 2 增到 3；斜率每增加 1 单位的预测就增加 1，因此 $h=1$。你也可以把它想成曲线的弯曲程度。

如果学过求导，可以这样核对：令预测为 $p$、真实值为 $y$，$L(p)=\frac12(p-y)^2$，对 $p$ 求一次导数得 $g=p-y$，再求一次导数得 $h=1$。还没学过求导也没关系：先记住在**这个损失**中，$g=\text{预测}-\text{真实}$，$h=1$，并看清修正的方向。别的损失不一定如此。

### 第二步：一组瓜共用多少修正？

现在有四个瓜，真实甜度分别是 **6、8、10、12**，当前都预测为 9。根据刚才的规则，每个瓜的 $g$ 是 `[3, 1, -1, -3]`，$h$ 都是 1。假设按某个候选问题分开：左边是前两个瓜，右边是后两个瓜。

一个叶子里的瓜共用同一个修正量。先把这片叶子里的梯度相加得到 $G$，把二阶信息相加得到 $H$。例如左边：$G_L=3+1=4$、$H_L=1+1=2$；右边：$G_R=-4$、$H_R=2$。

XGBoost 在此简化设定下，为该叶子估算的修正量为 $w^*=-G/(H+\lambda)$。其中 $\lambda$ 是对大叶权的 L2 惩罚；$\lambda$ 越大，分母越大，这组瓜的修正幅度就越保守。取 $\lambda=1$，左叶修正 $-4/3$，右叶修正 $+4/3$：左边整体预测偏高，需要往下调；右边整体偏低，需要往上调。**这是叶子给出的修正，不是最终甜度**；加入模型时还可乘学习率。

这里的叶权公式来自对新树损失的二阶近似，并仅展示 L2 正则、无 L1 正则的情况。想先掌握流程，可以把求解公式当成“根据这一叶的 $G$、$H$ 和惩罚，计算一份共用修正”的工具；下方代码会代入验证。

### 第三步：多分一片叶子值不值？

把四瓜留在同一组时，正负梯度互相抵消，父节点 $G=0$、$H=4$，共同修正为 0。分成左右两组后，左边可以下调、右边可以上调，模型有机会拟合得更好。但是树越复杂也越可能记住噪声，所以还要给新增叶子收取一个复杂度代价 $\gamma$。

![四瓜分组的叶权和增益计算：比较未分裂与分裂，并展示不同 gamma 的结果](../assets/watermelon_xgb_gain.png)

在只考虑 L2 的简化公式中，先算每组的 $G^2/(H+\lambda)$；这反映该组利用一份叶修正改善目标的潜力。**分裂带来的收益**是“左组潜力 + 右组潜力 − 原组潜力”的一半，再减新增叶子的代价：

$$\mathrm{Gain}=\tfrac12\left[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{G^2}{H+\lambda}\right]-\gamma$$

本例左项和右项都是 $16/3$，父项是 $0$：

- 若 $\gamma=0$，收益是 $16/3>0$，这个候选分裂在该简化目标下值得考虑。
- 若 $\gamma=6$，收益是 $16/3-6<0$，改善抵不过新增叶子的代价，按该标准就不应分裂。

训练时还会比较**其他候选问题**，并检查最小叶权等约束；收益为正不保证它最终一定被选中。这里的 $\gamma$ 是一分为二**新增的一片叶子**所带来的代价，不是改变甜度的数值。

![四瓜梯度、二阶信息、叶权与分裂收益汇总](../assets/watermelon_xgboost_mechanics.png)

试着用下格复算：增大 $\lambda$ 看两片叶子的修正怎样变化；再令 $\gamma=6$，看分裂收益是否转为负数。XGBoost 是一个软件库；它的树模型属于梯度提升。二阶信息、正则化和并行计算不是它独有的特性。

In [ ]:
# 【练习】平方误差的 XGBoost 风格演算；非 XGBoost 完整实现。
y_example = np.array([6., 8., 10., 12.]); pred_before = np.full(4, 9.)
g = pred_before - y_example; h = np.ones_like(g)
lambda_l2, gamma = 1., 0.
def leaf_weight(g_values, h_values, l2):
    return -np.sum(g_values) / (np.sum(h_values) + l2)
def split_gain(g_left, h_left, g_right, h_right, l2, gamma):
    gl, hl = np.sum(g_left), np.sum(h_left); gr, hr = np.sum(g_right), np.sum(h_right)
    return .5*(gl**2/(hl+l2) + gr**2/(hr+l2) - (gl+gr)**2/(hl+hr+l2)) - gamma
wl = leaf_weight(g[:2], h[:2], lambda_l2); wr = leaf_weight(g[2:], h[2:], lambda_l2)
gain = split_gain(g[:2], h[:2], g[2:], h[2:], lambda_l2, gamma)
np.testing.assert_allclose([leaf_weight(g[:2], h[:2], 1.), leaf_weight(g[2:], h[2:], 1.), split_gain(g[:2], h[:2], g[2:], h[2:], 1., 0.)], [-4/3, 4/3, 16/3])
print('g:', g, '；h:', h)
print('左叶权、右叶权、分裂增益：', round(wl,3), round(wr,3), round(gain,3))

分类任务中的导数针对模型的**原始分数**；新树不是直接输出最终类别概率。二阶近似、正则化、缺失值处理和并行计算也并非 XGBoost 独有。不同提升轮次仍有先后依赖，一轮内部的某些计算可以并行。[XGBoost 官方原理教程](https://xgboost.readthedocs.io/en/stable/tutorials/model.html)；[scikit-learn 集成方法](https://scikit-learn.org/stable/modules/ensemble.html)。

### 自测 6：叶权和正则化

In [ ]:
display(lesson_quiz('xgb'))

## 8. 回头看：这些方法是什么关系？

![从单树到随机森林、梯度提升树及 XGBoost 树模型的关系](../assets/watermelon_learning_map.png)

| 问题 | 单棵树 | 随机森林 | 梯度提升树 / XGBoost 树模型 |
| --- | --- | --- | --- |
| 树怎么来 | 对当前一组瓜反复提问 | 每棵看不同的抽样，节点选不同候选特征 | 新树根据当前模型的误差或损失方向训练 |
| 结果怎么合并 | 沿一条路径到叶子 | 多棵树预测后平均或合并 | 从初始预测开始，逐轮加修正 |
| 需要留意什么 | 太深容易记住噪声 | 各树仍可能一起犯错 | 树数、学习率和树复杂度都影响结果 |

有更多树或更复杂的算法，并不保证对新瓜更准。下一节把几种方法放到**同一批模拟瓜、同一甜度目标**上比较。

## 9. 同一道题比较，并留出最后评估

此前分类模型预测好坏，回归模型预测甜度，准确率与均方误差不能直接比较。现在各模型都使用三个输入、同一训练集和验证集预测**模拟甜度**；甜度标签本身不作为输入。参数先固定，结果只是这组模拟瓜的一次观察。

验证集用于比较；选定方案后，才打开从未参与选择的测试集评估。真实种植数据还需要考虑采样方式和时间顺序。

In [ ]:
regressors = {
    'Decision tree': DecisionTreeRegressor(max_depth=4, random_state=42),
    'Random forest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=-1),
    'Gradient boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=.08, max_depth=2, random_state=42),
}
validation_scores = {}
for name, estimator in regressors.items():
    estimator.fit(X_train, sweet_train)
    validation_scores[name] = mean_squared_error(sweet_valid, estimator.predict(X_valid))
    print(f'{name:18s} 验证 MSE = {validation_scores[name]:.3f}')

### 可选：使用 XGBoost 库

没有安装 XGBoost 时可以跳过，课程其他部分仍可运行。已安装时将它作为同一任务的候选；不同库的参数不完全等价，不能只凭这一次对比断言优劣。

In [ ]:
import importlib.util
if importlib.util.find_spec('xgboost') is None:
    print('未安装 xgboost，跳过可选对照。')
else:
    from xgboost import XGBRegressor
    xgb_model = XGBRegressor(n_estimators=100, max_depth=2, learning_rate=.08,
                             objective='reg:squarederror', random_state=42, n_jobs=1)
    xgb_model.fit(X_train, sweet_train)
    regressors['XGBoost'] = xgb_model
    validation_scores['XGBoost'] = mean_squared_error(sweet_valid, xgb_model.predict(X_valid))
    print(f"XGBoost 验证 MSE = {validation_scores['XGBoost']:.3f}")

### 验证集选定后，才打开测试集

选择验证误差最低的候选，仅用它对测试集计算一次误差。不要根据测试结果反复换模型；新的模型选择需要保留新的独立数据作最终评估。

In [ ]:
best_name = min(validation_scores, key=validation_scores.get)
best_model = regressors[best_name]
test_mse = mean_squared_error(sweet_test, best_model.predict(X_test))
print(f'验证集选出的模型：{best_name}')
print(f'最终测试 MSE：{test_mse:.3f}')

### 自测 7：模型评估

In [ ]:
display(lesson_quiz('evaluation'))

## 带走什么？

- **决策树**：找出一个能把已知瓜分得更纯的问题，在子组里继续问；回归树可让叶子输出组内平均甜度。
- **随机森林**：反复有放回抽样、随机选候选特征，分别训练不同的树，再合并判断。
- **梯度提升树**：把多棵树的修正按顺序相加；平方误差回归可用残差理解，其他损失需看对应负梯度。
- **XGBoost 的树模型**：属于梯度提升实现；利用目标的一阶、二阶信息和复杂度惩罚选择新树。这里的计算是教学简化，不是完整实现。

试着不看答案解释：① 为什么图中的纹理划分优于色泽？② 三棵树的概率怎样合并？③ 第一轮修正以后，第二轮拟合什么？④ 增大正则化参数会怎样影响叶权？⑤ 为什么不能用甜度本身作为预测甜度的输入？